In [5]:
import numpy as np


def run_data_imputation_pipeline(seed: int = 33) -> dict:
    """Generates synthetic matrix data with missing values (NaNs),
    imputes NaNs using column means, and extracts variance metrics.
    """
    np.random.seed(seed)

    # 1. Generate synthetic budget vs spend matrix (200 projects, 2 features)
    raw_data = np.random.randint(100, 1000, size=(200, 2)).astype(float)

    # 2. Inject missing values (NaNs) randomly across ~10% of values
    nan_mask = np.random.choice([True, False], size=raw_data.shape, p=[0.1, 0.9])
    raw_data[nan_mask] = np.nan

    # 3. Compute column-wise means while ignoring NaNs
    col_means = np.nanmean(raw_data, axis=0)

    # 4. Vectorized imputation: replace NaNs with corresponding column mean
    cleaned_data = raw_data.copy()
    missing_indices = np.isnan(cleaned_data)
    cleaned_data[missing_indices] = col_means[np.where(missing_indices)[1]]

    # 5. Extract analytical statistics via boolean matrix indexing
    budget_var = float(np.var(cleaned_data[:, 0]))
    spend_var = float(np.var(cleaned_data[:, 1]))

    # Filter projects where actual spend (col 1) strictly exceeds budget (col 0)
    overbudget_projects = cleaned_data[cleaned_data[:, 0] < cleaned_data[:, 1]]

    return {
        "raw_data": raw_data,
        "imputed_data": cleaned_data,
        "column_means": col_means,
        "budget_variance": budget_var,
        "spend_variance": spend_var,
        "overbudget_count": len(overbudget_projects),
    }


# --- Runnable Execution Block ---
if __name__ == "__main__":
    results = run_data_imputation_pipeline()

    print("=== NUMPY IMPUTATION PIPELINE OUTPUT ===")
    print(f"Calculated Column Means (Budget, Spend): {results['column_means']}")
    print(f"Budget Variance:                        {results['budget_variance']:.2f}")
    print(f"Spend Variance:                         {results['spend_variance']:.2f}")
    print(f"Total Over-budget Projects Identified:  {results['overbudget_count']}")

    print("\nFirst 5 Cleaned Data Rows [Budget, Spend]:")
    print(results["imputed_data"][:5])

=== NUMPY IMPUTATION PIPELINE OUTPUT ===
Calculated Column Means (Budget, Spend): [535.1299435  520.00549451]
Budget Variance:                        60961.01
Spend Variance:                         61396.25
Total Over-budget Projects Identified:  87

First 5 Cleaned Data Rows [Budget, Spend]:
[[120.         491.        ]
 [828.         678.        ]
 [758.         520.00549451]
 [157.         295.        ]
 [202.         161.        ]]
